In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# ============================================================
# 1. Ground truth SDF
# ============================================================
def hash2_torch(p):
    x = torch.sin(p[..., 0] * 12.9898 + p[..., 1] * 78.233) * 43758.5453
    return x - torch.floor(x)

def value_noise_2d_torch(x, z):
    xi, zi = torch.floor(x), torch.floor(z)
    xf, zf = x - xi, z - zi
    u = xf * xf * (3 - 2 * xf)
    v = zf * zf * (3 - 2 * zf)
    a = hash2_torch(torch.stack([xi,     zi    ], -1))
    b = hash2_torch(torch.stack([xi + 1, zi    ], -1))
    c = hash2_torch(torch.stack([xi,     zi + 1], -1))
    d = hash2_torch(torch.stack([xi + 1, zi + 1], -1))
    return a*(1-u)*(1-v) + b*u*(1-v) + c*(1-u)*v + d*u*v

def fbm_torch(x, z, octaves=5):
    f, amp, total, norm = 1.0, 1.0, 0.0, 0.0
    for _ in range(octaves):
        total = total + amp * value_noise_2d_torch(x * f, z * f)
        norm += amp
        f *= 2.0
        amp *= 0.5
    return total / norm

def ridged_torch(x, z, octaves=5):
    f, amp, total, norm = 1.0, 1.0, 0.0, 0.0
    for _ in range(octaves):
        n = value_noise_2d_torch(x * f, z * f)
        total = total + amp * (1.0 - torch.abs(2.0*n - 1.0))
        norm += amp
        f *= 2.0
        amp *= 0.5
    return total / norm

def gt_sdf_torch(p):
    x, y, z = p[..., 0], p[..., 1], p[..., 2]
    h = 0.3 * fbm_torch(x * 0.8, z * 0.8, octaves=4) - 0.2
    h = h + 0.05 * ridged_torch(x * 3.0, z * 3.0, octaves=3)
    sdf_ground = y - h
    cx = 0.6 * torch.sin(z * 0.5) + 0.3 * torch.sin(z * 1.3)
    dx = x - cx
    half_w = 0.4 + 0.15 * torch.sin(z * 0.7)
    horizontal = torch.abs(dx) - half_w
    canyon_depth = 0.8
    vertical = y - (h - canyon_depth)
    inside = torch.maximum(horizontal, -vertical)
    return torch.maximum(sdf_ground, -inside)

# ============================================================
# 2. ReLU MLP with frequency encoding 
# ============================================================
NUM_FREQS = 4
HIDDEN = 16
ENC_DIM = 3 + 6 * NUM_FREQS  # 39

def freq_encode(x, num_freqs=NUM_FREQS):
    out = [x]
    for i in range(num_freqs):
        f = (2.0 ** i) * np.pi
        out.append(torch.sin(x * f))
        out.append(torch.cos(x * f))
    return torch.cat(out, dim=-1)

class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(ENC_DIM, HIDDEN)
        self.fc2 = nn.Linear(HIDDEN, HIDDEN)
        self.fc3 = nn.Linear(HIDDEN, 1)
    def forward(self, x):
        x = freq_encode(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x).squeeze(-1)

model = TinyMLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(f"Model params: {sum(p.numel() for p in model.parameters())}")

# ============================================================
# 3. Pre-generate training data on GPU
# ============================================================
print("Generating training data on GPU...")
N_UNIFORM = 500_000

pts_uniform = torch.rand(N_UNIFORM, 3, device=device) * 4 - 2

pts_pool = torch.rand(N_UNIFORM * 16, 3, device=device) * 4 - 2
sdf_pool = gt_sdf_torch(pts_pool)
mask = sdf_pool.abs() < 0.15
pts_near = pts_pool[mask]
N_NEAR = min(len(pts_near), N_UNIFORM * 2)
pts_near = pts_near[:N_NEAR]
print(f"Got {len(pts_near)} near-surface samples")

pts_all = torch.cat([pts_uniform, pts_near])
sdf_all = gt_sdf_torch(pts_all)
print(f"Total training samples: {len(pts_all)}")

# ============================================================
# 4. Training with Huber loss + eikonal regularization
# ============================================================
BATCH = 16384
N_STEPS = 8000
HUBER_DELTA = 0.05
EIKONAL_WEIGHT = 0.05  # weight for ||∇sdf|| - 1 penalty

t0 = time.time()
for step in range(N_STEPS):
    idx = torch.randint(0, len(pts_all), (BATCH,), device=device)
    p, s = pts_all[idx], sdf_all[idx]
    
    # Main loss: Huber (smooth L1) — robust like L1 but smooth like L2 near 0
    pred = model(p)
    diff = pred - s
    abs_diff = diff.abs()
    huber = torch.where(
        abs_diff < HUBER_DELTA,
        0.5 * diff * diff / HUBER_DELTA,
        abs_diff - 0.5 * HUBER_DELTA
    )
    fit_loss = huber.mean()
    
    # Eikonal regularization: encourage ||∇sdf|| ≈ 1
    # Sample a small set of random points for gradient computation
    N_EIK = 2048
    p_eik = (torch.rand(N_EIK, 3, device=device) * 4 - 2).requires_grad_(True)
    s_eik = model(p_eik)
    grad = torch.autograd.grad(s_eik.sum(), p_eik, create_graph=True)[0]
    grad_norm = grad.norm(dim=-1)
    eik_loss = ((grad_norm - 1.0) ** 2).mean()
    
    loss = fit_loss + EIKONAL_WEIGHT * eik_loss
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    
    if step % 500 == 0:
        with torch.no_grad():
            raw = (pred - s).abs().mean()
            near_mask = s.abs() < 0.15
            near_err = (pred[near_mask] - s[near_mask]).abs().mean() if near_mask.any() else torch.tensor(0.0)
        print(f"step {step:5d}: total={loss.item():.5f}  fit={fit_loss.item():.5f}  "
              f"eik={eik_loss.item():.4f}  raw={raw.item():.5f}  near={near_err.item():.5f}")

print(f"\nTraining took {time.time()-t0:.1f}s")
torch.save(model.state_dict(), 'canyon_mlp.pt')
print("Saved canyon_mlp.pt")

# ============================================================
# 5. Visualize quality
# ============================================================
N = 256
xs = torch.linspace(-2, 2, N, device=device)
zs = torch.linspace(-2, 2, N, device=device)
X, Z = torch.meshgrid(xs, zs, indexing='xy')
Y = torch.full_like(X, -0.1)
P = torch.stack([X, Y, Z], dim=-1).reshape(-1, 3)

with torch.no_grad():
    gt_slice = gt_sdf_torch(P).reshape(N, N).cpu().numpy()
    pred_slice = model(P).reshape(N, N).cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(gt_slice, cmap='RdBu', vmin=-1, vmax=1, origin='lower')
ax[0].set_title('Ground Truth')
ax[1].imshow(pred_slice, cmap='RdBu', vmin=-1, vmax=1, origin='lower')
ax[1].set_title('MLP Prediction (ReLU + Eikonal)')
err = pred_slice - gt_slice
ax[2].imshow(err, cmap='RdBu', vmin=-0.1, vmax=0.1, origin='lower')
ax[2].set_title(f'Error (max={np.abs(err).max():.3f})')
plt.tight_layout()
plt.savefig('mlp_quality.png', dpi=120)
print("Saved mlp_quality.png")

# ============================================================
# 6. Test points
# ============================================================
test_pts = torch.tensor([
    [ 0.0,  0.0,  0.0],
    [ 1.0, -0.5,  0.5],
    [-1.5,  0.2, -1.0],
    [ 0.5,  0.8,  1.5],
    [-0.3, -0.3, -0.3],
], device=device)
with torch.no_grad():
    out = model(test_pts).cpu().numpy()
print("\n=== Test points for GLSL verification ===")
for p, o in zip(test_pts.cpu().numpy(), out):
    print(f"  p=({p[0]:+.3f},{p[1]:+.3f},{p[2]:+.3f})  expected SDF = {o:+.6f}")

# ============================================================
# 7. Export GLSL weights
# ============================================================
def export_weights_glsl(model, path='weights.glsl'):
    sd = model.state_dict()
    lines = [
        "// Auto-generated MLP weights for canyon SDF (ReLU + freq encoding)",
        f"// NUM_FREQS = {NUM_FREQS}, HIDDEN = {HIDDEN}, ENC_DIM = {ENC_DIM}",
        ""
    ]
    for name, tensor in sd.items():
        flat = tensor.cpu().numpy().flatten()
        glsl_name = name.replace('.', '_').upper()
        lines.append(f"const float {glsl_name}[{len(flat)}] = float[](")
        for i in range(0, len(flat), 6):
            chunk = flat[i:i+6]
            row = ", ".join(f"{v:+.6f}" for v in chunk)
            comma = "," if i+6 < len(flat) else ""
            lines.append(f"    {row}{comma}")
        lines.append(f"); // shape = {tuple(tensor.shape)}")
        lines.append("")
    with open(path, 'w') as f:
        f.write('\n'.join(lines))
    print(f"\nWrote {path}")

export_weights_glsl(model)

print("\n=== Constants for GLSL ===")
print(f"const int NUM_FREQS = {NUM_FREQS};")
print(f"const int ENC_DIM   = {ENC_DIM};")
print(f"const int HIDDEN    = {HIDDEN};")



Using device: cuda
Model params: 737
Generating training data on GPU...
Got 838911 near-surface samples
Total training samples: 1338911
step     0: total=0.36685  fit=0.35224  eik=0.2922  raw=0.37562  near=0.10718
step   500: total=0.02223  fit=0.01936  eik=0.0574  raw=0.03503  near=0.02192
step  1000: total=0.01166  fit=0.00994  eik=0.0344  raw=0.02334  near=0.01662
step  1500: total=0.00870  fit=0.00736  eik=0.0269  raw=0.01949  near=0.01401
step  2000: total=0.00646  fit=0.00533  eik=0.0226  raw=0.01652  near=0.01271
step  2500: total=0.00552  fit=0.00463  eik=0.0179  raw=0.01553  near=0.01230
step  3000: total=0.00492  fit=0.00403  eik=0.0177  raw=0.01446  near=0.01180
step  3500: total=0.00477  fit=0.00384  eik=0.0185  raw=0.01419  near=0.01151
step  4000: total=0.00408  fit=0.00328  eik=0.0160  raw=0.01303  near=0.01044
step  4500: total=0.00381  fit=0.00307  eik=0.0149  raw=0.01280  near=0.01044
step  5000: total=0.00362  fit=0.00288  eik=0.0149  raw=0.01250  near=0.01037
step  